# Spectral analysis with the Fourier Transform

This tutorial demonstrates how to use the Fourier Transform methods in **earthkit-transforms** to analyse a timeseries in the frequency domain. We will:

1. compute the forward FFT of an hourly timeseries with `temporal.fft`,
2. inspect the power spectrum and identify the dominant (diurnal) cycle,
3. centre the spectrum with `fftshift`,
4. apply a simple low-pass filter and reconstruct a smoothed signal with `temporal.ifft`, and
5. verify that the forward and inverse transforms round-trip.

The transforms use the `fft` extension of the array namespace of the input data (the Python array API standard), so they run on the native backend of the data and return `xarray` objects.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from earthkit import data as ekd
from earthkit import transforms as ekt

# Hourly 2m temperature timeseries (72 hourly steps over 3 days) for two locations
ds = ekd.from_source("sample", "era5-timeseries-multiple.nc").to_xarray()
da = ds["t2m"].sel(location="Reading")

da.plot.line(x="valid_time", marker="o")
plt.title("Hourly 2m temperature at Reading")

## 1. Forward transform

`temporal.fft` detects the time dimension (`valid_time`) automatically and returns a complex-valued result indexed by a `frequency` dimension. Because the data are sampled hourly, the frequency coordinate is derived in Hz (cycles per second).

In [ ]:
spectrum = ekt.temporal.fft(da)

spectrum

## 2. Power spectrum and the dominant cycle

The squared amplitude of each frequency component gives the power spectrum. We keep the positive frequencies and convert each frequency (Hz) to a period in hours to make the result easier to interpret.

In [ ]:
positive = spectrum.where(spectrum["frequency"] > 0, drop=True)
power = (np.abs(positive) ** 2).rename("power")

# Convert frequency in Hz to a period in hours
period_hours = 1.0 / (positive["frequency"] * 3600.0)
power = power.assign_coords(period_hours=("frequency", period_hours.data))

dominant_period = float(power["period_hours"][power.argmax("frequency")].item())
print(f"Dominant period: {dominant_period:.1f} hours")

power.plot.line(x="period_hours", marker="o")
plt.xlabel("Period (hours)")
plt.title("Power spectrum")

The strongest peak is close to 24 hours, reflecting the diurnal cycle of near-surface temperature.

## 3. Centre the spectrum with `fftshift`

The raw FFT output orders the frequencies from zero upwards and then wraps around to the negative frequencies. `fftshift` reorders the data (and its frequency coordinate) so that the zero-frequency component is in the centre, which is often more convenient for plotting. The generic spectrum helpers live in the `earthkit.transforms._fourier` module.

In [ ]:
shifted = ekt._fourier.fftshift(spectrum, dim="frequency")

np.abs(shifted).plot.line(x="frequency", marker="o")
plt.title("Amplitude spectrum (zero frequency centred)")

## 4. Low-pass filter and inverse transform

We can filter in the frequency domain and transform back to the time domain. Here we keep only the low-frequency components (periods of 12 hours or longer) by setting the higher frequencies to zero, then reconstruct a smoothed signal with `temporal.ifft`.

In [ ]:
# Keep components with a period of 12 hours or longer
cutoff_hz = 1.0 / (12 * 3600.0)
filtered = spectrum.where(np.abs(spectrum["frequency"]) <= cutoff_hz, 0)

smoothed = ekt.temporal.ifft(filtered, time_dim="valid_time", time_coord=da["valid_time"].values).real

da.plot.line(x="valid_time", marker="o", label="original")
smoothed.plot.line(x="valid_time", label="low-pass filtered")
plt.legend()
plt.title("Low-pass filtered temperature")

## 5. Round-trip verification

Applying the inverse transform to the full spectrum recovers the original signal (up to floating-point precision).

In [ ]:
restored = ekt.temporal.ifft(spectrum, time_dim="valid_time", time_coord=da["valid_time"].values).real

print("Maximum absolute difference:", float(np.abs(restored - da).max()))

## Beyond the time dimension

The same transforms are available generically in the `earthkit.transforms._fourier` module, where the dimension is specified explicitly (for example `earthkit.transforms._fourier.fft(dataarray, dim="longitude")`). That module also provides the real-valued transforms (`rfft`/`irfft`), the n-dimensional transforms (`fftn`/`ifftn`/`rfftn`/`irfftn`), the sample-frequency helpers (`fftfreq`/`rfftfreq`) and the spectrum shifts (`fftshift`/`ifftshift`) from the Python array API standard.